# 01-03 손실과 경사 하강법

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 데이터: y = 2x + 1 근처에 흩어진 점

In [ ]:
torch.manual_seed(0)
x = torch.linspace(-1, 1, 30)  # (30,)
y = 2 * x + 1 + 0.3 * torch.randn(30)  # (30,)

plt.figure(figsize=(4.5, 3.5))
plt.scatter(x, y, color="tab:blue", edgecolor="k")
plt.plot(x, 0.5 * x + 0.0, color="gray", ls="--", label="w=0.5, b=0")
plt.plot(x, 2 * x + 1, color="tab:orange", label="w=2, b=1")
plt.legend()
plt.show()

## 2. 손실: 예측이 얼마나 틀렸는지 숫자 하나로

In [ ]:
def predict(w, b):
    return w * x + b  # (30,)

def mse(w, b):
    return ((predict(w, b) - y) ** 2).mean()

print("w=0.5, b=0 손실:", mse(0.5, 0.0).item())
print("w=2.0, b=1 손실:", mse(2.0, 1.0).item())

In [ ]:
ws = torch.linspace(-1, 5, 121)
bs = torch.linspace(-2, 4, 121)
L = torch.stack([torch.stack([mse(w, b) for w in ws]) for b in bs])  # (121, 121)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ws, [mse(w, 1.0).item() for w in ws], color="tab:red")
axes[0].set_xlabel("w (b = 1)")
axes[0].set_ylabel("loss")
axes[0].set_title("loss vs w")
cs = axes[1].contourf(ws, bs, L, levels=30, cmap="Reds")
plt.colorbar(cs, ax=axes[1])
axes[1].set_xlabel("w")
axes[1].set_ylabel("b")
axes[1].set_title("loss over (w, b)")
plt.show()

## 3. 기울기 구하기: `backward()`

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

loss = mse(w, b)
loss.backward()  # 손실을 w, b로 미분한 값을 계산해요
print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())

In [ ]:
# 직접 확인: w를 아주 조금 바꿨을 때 손실이 얼마나 바뀌는지
eps = 1e-3
numeric = (mse(0.5 + eps, 0.0) - mse(0.5 - eps, 0.0)) / (2 * eps)
print("숫자로 구한 기울기:", numeric.item())

## 4. 경사 하강법: 기울기 반대 방향으로 조금씩

In [ ]:
def gradient_descent(lr, steps=50, w0=0.5, b0=0.0):
    w = torch.tensor(w0, requires_grad=True)
    b = torch.tensor(b0, requires_grad=True)
    history = []
    for step in range(steps):
        loss = mse(w, b)
        history.append((w.item(), b.item(), loss.item()))
        loss.backward()
        with torch.no_grad():  # 파라미터를 고치는 계산은 기록하지 않아요
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()  # 기울기는 누적되므로 매번 0으로 지워요
        b.grad.zero_()
    return history

history = gradient_descent(lr=0.1)
for step in [0, 10, 20, 49]:
    w_, b_, l_ = history[step]
    print(f"step {step:2d}: w={w_:.3f}, b={b_:.3f}, loss={l_:.4f}")

In [ ]:
path = torch.tensor(history)  # (50, 3)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contourf(ws, bs, L, levels=30, cmap="Reds")
axes[0].plot(path[:, 0], path[:, 1], "o-", color="tab:orange", ms=3)
axes[0].set_xlabel("w")
axes[0].set_ylabel("b")
axes[0].set_title("path of (w, b)")
axes[1].plot(path[:, 2], color="tab:red")
axes[1].set_xlabel("step")
axes[1].set_ylabel("loss")
axes[1].set_title("loss curve")
plt.show()

## 5. 학습률이 너무 작거나 크면

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lr in zip(axes, [0.01, 0.1, 1.05]):
    p = torch.tensor(gradient_descent(lr=lr, steps=20))
    ax.contourf(ws, bs, L, levels=30, cmap="Reds")
    ax.plot(p[:, 0].clamp(-1, 5), p[:, 1].clamp(-2, 4), "o-", color="tab:orange", ms=3)
    ax.set_title(f"lr = {lr}, final loss = {p[-1, 2]:.3g}")
    ax.set_xlabel("w")
    ax.set_ylabel("b")
plt.show()